In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/elastic/lib/python3.10/site-packages/IPython/core/extensions.py:151: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  mod.load_ipython_extension(self.shell)


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/annotated/checkpoints/pre_cell_10.pickle

trying: ['SEP_DOLLAR', 'CONVERT_TO_CAT', 'VARIABLE_FILES', 'AUTO_ADJUST_LEARNING_RATE']
me:  2
me:  2
me:  2
me:  2
trying: ['SEP_DOLLAR', 'CONVERT_TO_CAT', 'VARIABLE_FILES', 'AUTO_ADJUST_LEARNING_RATE']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA', 'ENABLE_BREAKPOINT']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA', 'ENABLE_BREAKPOINT']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'ENABLE_BREAKPOINT', 'SEP_COMMA', 'REGRESSOR']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'ENABLE_BREAKPOINT', 'SEP_COMMA', 'REGRESSOR']
me:  2
me:  2
me:  2
me:  2
trying: ['SEP_DOLLAR', 'CONVERT_TO_CAT', 'VARIABLE_FILES', 'AUTO_ADJUST_LEARNING_RATE']
me:  2
me:  2
me:  2
me:  2
trying: ['SEP_DOLLAR', 'CONVERT_TO_CAT', 'VARIABLE_FILES', 'AUTO_ADJUST_LEARNING_RATE']
me:  2
me:  2
me:  2
me:  2
trying: ['FASTAI_LEARNING_RATE']
me:  2
trying: ['random']
me:  1
trying: ['df']


me:  6
trying: ['factor']
me:  5
trying: ['np']
me:  1
trying: ['shutil']
me:  1
trying: ['pd']
me:  1
trying: ['traceback']
me:  1
trying: ['col']
me:  6
trying: ['PROJECT_NAME']
me:  2
trying: ['warnings']
me:  0
trying: ['PARAM_DIR']
me:  3


Declaring variable warnings
Declaring variable random
Declaring variable np
Declaring variable shutil
Declaring variable pd
Declaring variable traceback
Declaring variable SEP_DOLLAR
Declaring variable CONVERT_TO_CAT
Declaring variable VARIABLE_FILES
Declaring variable AUTO_ADJUST_LEARNING_RATE
Declaring variable SEP_DOLLAR
Declaring variable CONVERT_TO_CAT
Declaring variable VARIABLE_FILES
Declaring variable AUTO_ADJUST_LEARNING_RATE
Declaring variable SHUFFLE_DATA
Declaring variable REGRESSOR
Declaring variable SEP_COMMA
Declaring variable ENABLE_BREAKPOINT
Declaring variable SHUFFLE_DATA
Declaring variable REGRESSOR
Declaring variable SEP_COMMA
Declaring variable ENABLE_BREAKPOINT
Declaring variable SHUFFLE_DATA
Declaring variable ENABLE_BREAKPOINT
Declaring variable SEP_COMMA
Declaring variable REGRESSOR
Declaring variable SHUFFLE_DATA
Declaring variable ENABLE_BREAKPOINT
Declaring variable SEP_COMMA
Declaring variable REGRESSOR
Declaring variable SEP_DOLLAR
Declaring variable CONV

In [4]:
%%RecordEvent
%%time
### cell 10 ###

import os

target = ''
target_str = ''
targets = []

# Ensure PARAM_DIR and config files exist
os.makedirs(PARAM_DIR, exist_ok=True)
for fname in ('cats.txt','conts.txt','cols_to_delete.txt'):
    fpath = os.path.join(PARAM_DIR, fname)
    if not os.path.exists(fpath):
        open(fpath, 'w').close()

# Loop backwards through candidate target columns
for i in range(len(df.columns) - 1, 0, -1):
    col = df.columns[i]
    try:
        # Convert to numeric (non-numeric → NaN)
        df[col] = pd.to_numeric(df[col], errors='coerce')
        target = col
        target_str = target.replace('/', '-')
    except:
        continue
    print(f'Target Variable: {target}')

    # Drop duplicates
    df = df.drop_duplicates()
    # Optionally shuffle
    if SHUFFLE_DATA:
        df = df.sample(frac=1).reset_index(drop=True)

    # Convert boolean columns to uint8
    for c in df.columns:
        if pd.api.types.is_bool_dtype(df[c]):
            df[c] = df[c].astype('uint8')

    # Remove any columns listed for deletion
    with open(os.path.join(PARAM_DIR, 'cols_to_delete.txt'), 'r') as f:
        cols_to_delete = f.read().splitlines()
    for c in cols_to_delete:
        if c in df.columns:
            try:
                del df[c]
            except:
                pass

    # Fill remaining NaNs with zero
    df = df.fillna(0)

    # Auto-detect categorical vs continuous
    nunique = df.nunique()
    count = df.count()
    mask = nunique / count < 0.05
    cats = mask[mask].index.tolist()
    conts = mask[~mask].index.tolist()
    # Remove the target from those lists
    if target in cats:  cats.remove(target)
    if target in conts: conts.remove(target)

    # Re-ensure target is numeric
    try:
        df[target] = pd.to_numeric(df[target], errors='coerce')
    except:
        pass

    # If overriding via files
    if VARIABLE_FILES:
        with open(os.path.join(PARAM_DIR, 'cats.txt'), 'r') as f:
            cats = f.read().splitlines()
        with open(os.path.join(PARAM_DIR, 'conts.txt'), 'r') as f:
            conts = f.read().splitlines()

    # Convert continuous vars to numeric
    for c in conts:
        try:
            df[c] = pd.to_numeric(df[c], errors='coerce')
        except:
            print(f'Could not convert {c} to float.')

    # Extra breakpoint conversions
    if ENABLE_BREAKPOINT:
        cont_list = list(conts)
        for var in cont_list:
            try:
                df[var] = pd.to_numeric(df[var], errors='coerce')
            except:
                print(f'Could not convert {var} to float.')
                cont_list.remove(var)
                if CONVERT_TO_CAT:
                    cats.append(var)
                pass


Target Variable: Profit_processed


Target Variable: Discount_processed
Target Variable: Quantity_processed
Target Variable: Sales_processed
Target Variable: Sub-Category_processed


Target Variable: Category_processed
Target Variable: Region_processed
Target Variable: Postal Code_processed
Target Variable: State_processed
Target Variable: City_processed


Target Variable: Country_processed
Target Variable: Segment_processed
Target Variable: Ship Mode_processed
Target Variable: Profit
Target Variable: Discount


Target Variable: Quantity
Target Variable: Sales
Target Variable: Sub-Category
Target Variable: Category


Target Variable: Region
Target Variable: Postal Code
Target Variable: State
Target Variable: City


Target Variable: Country
Target Variable: Segment
CPU times: user 1.53 s, sys: 63 ms, total: 1.59 s
Wall time: 1.59 s


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/rewritten/cell_10/checkpoints/post_cell_10_try_5.pickle

migration speed (bps): 780821238.326457
---------------------------
variables to migrate:
factor 28
pd 72
SEP_DOLLAR 24
warnings 72
SHUFFLE_DATA 28
REGRESSOR 28
SEP_COMMA 28
random 72
AUTO_ADJUST_LEARNING_RATE 24
PROJECT_NAME 59
fpath 170
CONVERT_TO_CAT 24
conts 104
VARIABLE_FILES 24
target 56
FASTAI_LEARNING_RATE 24
count 48
ENABLE_BREAKPOINT 28
target_str 56
nunique 48
cont_list 104
col 56
var 65
f 208
df 48
np 72
mask 48
PARAM_DIR 151
shutil 72
cats 216
cols_to_delete 56
fname 67
i 28
traceback 72
targets 56
c 65
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/rewritten/cell_10/checkpoints/post_cell_10_try_5.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['df', 'PARAM_DIR']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['df', 'PARAM_DIR']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables ['df']
Intermediate variables ['factor']
Future variables ['PARAM_DIR']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables ['col', 'df']
Intermediate variables []
Future variables ['PARAM_DIR']
Modified dataframes
  df
    Input columns: set()
    Changed columns: set()
    Created columns: {'Segment_processed', 'City_processed', 'Ship Mode_processed', 'Region_processed', 'Quantity_processed', 'Discount_processed', 'State_processed', 'Category_processed', 'Sub-Category_processed', 'Profit_processed', 'Postal Code_processed', 'Country_processed', 'Sales_processed'}
    Deleted columns: set()
======= Cell 4 =======
In

In [7]:
with open(
    "/home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/opt_cell_exec_info_10_try_5.pkl",
    "wb",
) as f:
    pickle.dump(opt_cell_exec_info[10], f)

In [8]:
opt_output = Out.get(4)

In [9]:
df_opt = df
%LoadCheckpoint /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/annotated/checkpoints/post_cell_10.pickle
assert compare_df(df_opt, df)

import numpy as np
from elastic.core.common.pandas import is_type_styler

is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(
    orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_opt_output_array = isinstance(
    opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif (
    (is_orig_output_pd or is_opt_output_pd)
    and (is_orig_output_array or is_opt_output_array)
) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output

trying: ['AUTO_ADJUST_LEARNING_RATE', 'SEP_DOLLAR', 'VARIABLE_FILES', 'CONVERT_TO_CAT']
me:  2
me:  2
me:  2
me:  2
trying: ['AUTO_ADJUST_LEARNING_RATE', 'SEP_DOLLAR', 'VARIABLE_FILES', 'CONVERT_TO_CAT']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'ENABLE_BREAKPOINT', 'REGRESSOR', 'SEP_COMMA']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'ENABLE_BREAKPOINT', 'REGRESSOR', 'SEP_COMMA']
me:  2
me:  2
me:  2
me:  2
trying: ['ENABLE_BREAKPOINT', 'SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA']
me:  2
me:  2
me:  2
me:  2
trying: ['ENABLE_BREAKPOINT', 'SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA']
me:  2
me:  2
me:  2
me:  2
trying: ['AUTO_ADJUST_LEARNING_RATE', 'VARIABLE_FILES', 'SEP_DOLLAR', 'CONVERT_TO_CAT']
me:  2
me:  2
me:  2
me:  2
trying: ['AUTO_ADJUST_LEARNING_RATE', 'VARIABLE_FILES', 'SEP_DOLLAR', 'CONVERT_TO_CAT']
me:  2
me:  2
me:  2
me:  2
trying: ['focus_cont', 'cont', 'n', 'var']
me:  13
me:  13
me:  13
me:  13
trying: ['focus_cont', 'cont', 'n', 'var']
me:  13
me:  13
me

ValueError: Content of df1 and df2 do not match